# Isopropyl Alcohol

In [1]:
using Clapeyron, Metaheuristics, Printf

In [17]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
isopropylalcohol,60.096,3.0260,3.2279,215.0766,1,1
"""

assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
isopropylalcohol,H,isopropylalcohol,e,2000.0,0.03
"""

model = PCSAFT(["isopropylalcohol"], userlocations = [like_parameter, assoc_parameter])
print(model.params.epsilon_assoc.values)

Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2000.0]

In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_isopropylalcohol.csv")
fix_line_endings("rhol_isopropylalcohol.csv")

Fixed: satp_isopropylalcohol.csv
Fixed: rhol_isopropylalcohol.csv


In [4]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end

println(my_liquid_density(model, 293.15))
println(my_saturation_p(model, 273.15))

780.1464468243072
2911.6432761789033


In [21]:
toestimate = [
    Dict(
        :param => :segment,
        :lower => 1.0,
        :upper => 10.0,
        :guess => 3.
    ),
    Dict(
        :param => :epsilon,
        :lower => 100.,
        :upper => 500.,
        :guess => 250.
    ),
    Dict(
        :param => :sigma,
        :factor => 1e-10,
        :lower => 2.0,
        :upper => 7.0,
        :guess => 5.
    ),
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 5000.0,
        :guess   => 2400.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 1.0,
        :guess   => 0.4
    )
]

5-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 10.0, :param => :segment, :guess => 3.0, :lower => 1.0)
 Dict(:upper => 500.0, :param => :epsilon, :guess => 250.0, :lower => 100.0)
 Dict(:factor => 1.0e-10, :param => :sigma, :upper => 7.0, :guess => 5.0, :lower => 2.0)
 Dict(:upper => 5000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 2400.0, :lower => 0.0)
 Dict(:upper => 1.0, :param => :bondvol, :indices => (1, 1), :guess => 0.4, :lower => 0.0)

In [22]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_isopropylalcohol.csv",
        "satp_isopropylalcohol.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 1.5049873467903405


In [23]:
method = ECA(; options = Options(iterations = 10000000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([3.025974973402295, 215.07665905466166, 3.2279234296228325, 2216.77887479232, 0.021304564558238235], PCSAFT{BasicIdeal, Float64}("isopropylalcohol"))

In [24]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [25]:
aard_p   = calculate_AAD(model_opt, "satp_isopropylalcohol.csv",   my_saturation_p)


=== AAD: satp_isopropylalcohol.csv ===
Clapeyron Estimator  exp           calc          ARD%    
305.7800    9560.000000   9611.890364   0.5428  
308.5700    11320.000000  11258.140576  0.5465  
311.1200    12900.000000  12967.970144  0.5269  
314.0200    15270.000000  15177.031169  0.6088  
316.9700    17780.000000  17744.702434  0.1985  
319.6200    20470.000000  20356.458828  0.5547  
322.4300    23350.000000  23473.892816  0.5306  
324.8900    26590.000000  26524.608393  0.2459  
327.3400    29750.000000  29887.723003  0.4629  
330.3500    34430.000000  34502.534819  0.2107  
333.6700    40340.000000  40269.235413  0.1754  
339.2600    51700.000000  51786.247248  0.1668  
342.4100    59250.000000  59397.539289  0.2490  
345.8400    68840.000000  68713.000990  0.1845  
348.2300    76090.000000  75889.535377  0.2635  
350.7500    83990.000000  84111.483894  0.1446  
352.1900    89190.000000  89128.091409  0.0694  
355.4100    101300.000000  101234.165291  0.0650  
355.4200    101220

┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


0.30524408143089954

In [26]:
aard_p   = calculate_AAD(model_opt, "rhol_isopropylalcohol.csv",   my_liquid_density)


=== AAD: rhol_isopropylalcohol.csv ===
Clapeyron Estimator  exp           calc          ARD%    
293.1500    785.400000    785.485688    0.0109  
298.1500    781.100000    781.158306    0.0075  
303.1500    776.800000    776.789601    0.0013  
308.1500    772.400000    772.374581    0.0033  
313.1500    768.000000    767.908243    0.0119  
318.1500    763.400000    763.385552    0.0019  
323.1500    758.800000    758.801425    0.0002  
AARD = 0.0053%


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


0.005290327524191294